In [1]:
pip install transformers torch bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00


In [2]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
prompt = "Deep learning models understand language because"

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [3]:
def model_size_mb(model):
    total_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    return total_bytes / (1024 ** 2)


def measure_generation(model, tokenizer, prompt, max_new_tokens=30):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    start = time.time()
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    elapsed = time.time() - start
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return elapsed, text


In [4]:
# --------------------------------------------------
# Step 1: Load Full-Precision (FP32) Model
# --------------------------------------------------

print("===== Loading Full-Precision Model =====")
model_fp32 = AutoModelForCausalLM.from_pretrained(model_name)

size_fp32 = model_size_mb(model_fp32)
time_fp32, text_fp32 = measure_generation(model_fp32, tokenizer, prompt)

print(f"Model size (FP32):      {size_fp32:.2f} MB")
print(f"Generation time (FP32): {time_fp32:.2f} sec")
print("Sample output:", text_fp32)

del model_fp32
torch.cuda.empty_cache() if torch.cuda.is_available() else None


===== Loading Full-Precision Model =====


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Model size (FP32):      312.47 MB
Generation time (FP32): 2.07 sec
Sample output: Deep learning models understand language because they are able to understand the language and the language.





















In [5]:
print("\n===== Loading 8-bit Quantized Model =====")
bnb_config_8bit = BitsAndBytesConfig(load_in_8bit=True)

model_int8 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config_8bit,
    device_map="auto",
)

size_int8 = model_size_mb(model_int8)
time_int8, text_int8 = measure_generation(model_int8, tokenizer, prompt)

print(f"Model size (INT8):      {size_int8:.2f} MB")
print(f"Generation time (INT8): {time_int8:.2f} sec")
print("Sample output:", text_int8)

del model_int8
torch.cuda.empty_cache() if torch.cuda.is_available() else None



===== Loading 8-bit Quantized Model =====


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Model size (INT8):      190.97 MB
Generation time (INT8): 5.29 sec
Sample output: Deep learning models understand language because they are able to understand the language and the language.





















In [6]:
print("\n===== Loading 4-bit Quantized Model =====")
bnb_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config_4bit,
    device_map="auto",
)

size_int4 = model_size_mb(model_int4)
time_int4, text_int4 = measure_generation(model_int4, tokenizer, prompt)

print(f"Model size (INT4):      {size_int4:.2f} MB")
print(f"Generation time (INT4): {time_int4:.2f} sec")
print("Sample output:", text_int4)



===== Loading 4-bit Quantized Model =====


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model size (INT4):      170.72 MB
Generation time (INT4): 64.83 sec
Sample output: Deep learning models understand language because they are able to learn from the language.























In [ ]:
print("\n===== Comparison =====")
print(f"{'Precision':<10}{'Size (MB)':<15}{'Gen Time (sec)':<18}")
print(f"{'FP32':<10}{size_fp32:<15.2f}{time_fp32:<18.2f}")
print(f"{'INT8':<10}{size_int8:<15.2f}{time_int8:<18.2f}")
print(f"{'INT4':<10}{size_int4:<15.2f}{time_int4:<18.2f}")

print(f"\nSize reduction (INT8 vs FP32): {100 * (1 - size_int8 / size_fp32):.1f}%")
print(f"Size reduction (INT4 vs FP32): {100 * (1 - size_int4 / size_fp32):.1f}%")


In [7]:
print("\n===== Comparison =====")
print(f"{'Precision':<10}{'Size (MB)':<15}{'Gen Time (sec)':<18}")
print(f"{'FP32':<10}{size_fp32:<15.2f}{time_fp32:<18.2f}")
print(f"{'INT8':<10}{size_int8:<15.2f}{time_int8:<18.2f}")
print(f"{'INT4':<10}{size_int4:<15.2f}{time_int4:<18.2f}")

print(f"\nSize reduction (INT8 vs FP32): {100 * (1 - size_int8 / size_fp32):.1f}%")
print(f"Size reduction (INT4 vs FP32): {100 * (1 - size_int4 / size_fp32):.1f}%")



===== Comparison =====
Precision Size (MB)      Gen Time (sec)    
FP32      312.47         2.07              
INT8      190.97         5.29              
INT4      170.72         64.83             

Size reduction (INT8 vs FP32): 38.9%
Size reduction (INT4 vs FP32): 45.4%
